# Notebook 16 – Complete Preprocessing Workflow

**Dataset:** Reproducible synthetic telecom customer churn data. It includes missing values, duplicates, outliers, numeric and categorical features, and an imbalanced binary target.

**Key principle:** Split the data before learning imputation values, scaling parameters, category vocabularies, feature-selection rules, or resampling decisions. Fit transformations on training data only.

## 1. Imports and dataset creation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", None)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200
age = rng.normal(38, 12, n).clip(18, 80).round()
tenure = rng.integers(1, 73, n)
monthly_charges = (rng.normal(70, 22, n) + tenure * 0.08).clip(15, 160).round(2)
support_calls = rng.poisson(1.8, n)
contract = rng.choice(["Month-to-month", "One year", "Two year"], n, p=[0.58, 0.25, 0.17])
internet = rng.choice(["Fiber", "DSL", "No"], n, p=[0.48, 0.37, 0.15])
payment = rng.choice(["Card", "Bank transfer", "Electronic check"], n)
region = rng.choice(["North", "South", "East", "West"], n)
# Create an imbalanced target with plausible relationships
logit = -2.0 + 0.75*(contract == "Month-to-month") + 0.35*(support_calls >= 4) - 0.025*tenure + 0.012*(monthly_charges-70)
prob = 1/(1+np.exp(-logit))
churn = rng.binomial(1, prob)
df_raw = pd.DataFrame({
    "customer_id": [f"C{i:05d}" for i in range(n)],
    "age": age, "tenure_months": tenure, "monthly_charges": monthly_charges,
    "support_calls": support_calls, "contract": contract, "internet_service": internet,
    "payment_method": payment, "region": region, "churn": churn
})
# Introduce missingness, outliers, and exact duplicates for teaching
for col, frac in {"age":0.06, "monthly_charges":0.04, "contract":0.03, "payment_method":0.05}.items():
    idx = rng.choice(df_raw.index, int(n*frac), replace=False)
    df_raw.loc[idx, col] = np.nan
outlier_idx = rng.choice(df_raw.index, 8, replace=False)
df_raw.loc[outlier_idx, "monthly_charges"] = rng.uniform(350, 600, len(outlier_idx))
df_raw = pd.concat([df_raw, df_raw.sample(12, random_state=7)], ignore_index=True)
print("Raw dataset:", df_raw.shape)
df_raw.head()

## 2. Inspect dataset and identify data types

In [ ]:
print("Shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.sample(5, random_state=1))
df_raw.info()
display(df_raw.describe(include="all").T)

## 3. Identify missing values

In [ ]:
missing = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean()*100).round(2),
    "dtype": df_raw.dtypes.astype(str)
})
display(missing.sort_values("missing_percent", ascending=False))
sns.heatmap(df_raw.isna(), cbar=False); plt.title("Missing-value pattern"); plt.show()

## 4. Handle missing values – planning

Potential strategies:
- Numeric: median is robust to skew and outliers; mean is suitable for roughly symmetric data.
- Categorical: most frequent category or an explicit `"Unknown"` category.
- High missingness (for example, around 40%): investigate why values are missing, assess predictive value, consider an indicator, impute, or drop the feature only with domain justification.
- MCAR: missingness is unrelated to observed or unobserved values.
- MAR: missingness can be explained by observed variables.
- MNAR: missingness depends on the missing value itself or an unobserved factor.

The pipeline below learns imputation statistics from training data only.

## 5. Detect and handle duplicates

In [ ]:
print("Exact duplicates:", df_raw.duplicated().sum())
print("Duplicate customer IDs:", df_raw["customer_id"].duplicated().sum())
# Exact duplicate rows are removed for this example.
# In a real dataset, resolve conflicting records using business keys and timestamps.
df = df_raw.drop_duplicates().copy()
print("After exact duplicate removal:", df.shape)

## 6. Validate data

In [ ]:
# Example validation checks – adapt ranges and allowed categories to business rules.
assert df["customer_id"].notna().all()
assert df["churn"].dropna().isin([0, 1]).all()
assert df["tenure_months"].dropna().between(1, 72).all()
assert df["support_calls"].dropna().ge(0).all()
assert df["monthly_charges"].dropna().ge(0).all()
print("Basic validation checks passed.")

## 7. Detect outliers

In [ ]:
numeric_review = ["age","tenure_months","monthly_charges","support_calls"]
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3-q1
    return q1-1.5*iqr, q3+1.5*iqr

for col in numeric_review:
    lo, hi = iqr_bounds(df[col].dropna())
    count = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: lower={lo:.2f}, upper={hi:.2f}, flagged={count}")
df[numeric_review].plot(kind="box", subplots=True, layout=(1,4), figsize=(14,4), sharey=False)
plt.tight_layout(); plt.show()

## 8. Treat outliers – train-only clipping limits

For this demonstration, use percentile capping for `monthly_charges`. The limits are learned on the training split and then applied to the test split. Do not cap a feature automatically just because it is flagged.

## 9. Define features and target, then split

In [ ]:
# Identifier is retained separately for traceability, not used as a predictor.
X = df.drop(columns=["churn", "customer_id"])
y = df["churn"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train target proportions:\\n", y_train.value_counts(normalize=True))
print("Test target proportions:\\n", y_test.value_counts(normalize=True))

## 10. Build numeric and categorical preprocessing pipelines

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
], remainder="drop", verbose_feature_names_out=False)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 11. Outlier capping transformer

This small sklearn-compatible transformer learns percentile limits in `fit()` and applies them in `transform()`. It is placed inside the numeric pipeline, so limits are learned only from training data.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(X, self.lower, axis=0)
        self.upper_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", QuantileClipper(lower=0.01, upper=0.99)),
    ("scaler", StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
], remainder="drop", verbose_feature_names_out=False)

## 12. Encode categorical variables and scale numeric variables

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_ready = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_ready = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed train:", X_train_ready.shape)
print("Processed test:", X_test_ready.shape)
display(X_train_ready.head())

## 13. Handle class imbalance

First inspect the class ratio. Do not resample the full dataset before splitting. Options include class weights, random over/under-sampling, and SMOTE. SMOTE must be applied to training data only and should be used carefully with one-hot categorical data; for mixed numeric/categorical data consider `SMOTENC`.

This example uses `class_weight='balanced'` rather than synthetic resampling.

In [ ]:
print("Training class counts:")
display(y_train.value_counts())
print("Training class percentages:")
display((y_train.value_counts(normalize=True)*100).round(2))

## 14. Feature selection

In [ ]:
# Select up to 10 features using mutual information, learned on training data only.
k = min(10, X_train_ready.shape[1])
selector = SelectKBest(score_func=mutual_info_classif, k=k)
X_train_selected = selector.fit_transform(X_train_ready, y_train)
X_test_selected = selector.transform(X_test_ready)
selected_features = X_train_ready.columns[selector.get_support()].tolist()
print("Selected features:", selected_features)

## 15. Train a baseline model and evaluate

In [ ]:
model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
model.fit(X_train_selected, y_train)
pred = model.predict(X_test_selected)
proba = model.predict_proba(X_test_selected)[:, 1]

print(classification_report(y_test, pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix"); plt.show()

## 16. Build a complete sklearn preprocessing + feature-selection + model pipeline

This single pipeline ensures preprocessing and feature selection are fitted within each training fold when used with cross-validation. The custom clipper is included in the numeric branch.

In [ ]:
complete_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", QuantileClipper(0.01, 0.99)),
    ("scaler", StandardScaler())
])
complete_categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
complete_preprocessor = ColumnTransformer(transformers=[
    ("num", complete_numeric_pipeline, numeric_features),
    ("cat", complete_categorical_pipeline, categorical_features)
])

complete_pipeline = Pipeline(steps=[
    ("preprocessing", complete_preprocessor),
    ("feature_selection", SelectKBest(score_func=mutual_info_classif, k="all")),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
complete_pipeline.fit(X_train, y_train)
print("Complete pipeline test score:", round(complete_pipeline.score(X_test, y_test), 3))

## 17. Validate final ML-ready dataset

Validation checklist:
- No missing values in transformed matrices.
- Train and test have the same columns and column order.
- No identifier or target included in predictors.
- Imputers, scalers, encoders, clipping limits and selectors fitted on training data only.
- Target remains separate and has expected labels.
- No duplicate training records after the selected duplicate policy.

In [ ]:
assert X_train_ready.columns.equals(X_test_ready.columns)
assert not X_train_ready.isna().any().any()
assert not X_test_ready.isna().any().any()
assert "customer_id" not in X_train_ready.columns
assert "churn" not in X_train_ready.columns
assert set(y_train.unique()).issubset({0,1})
print("Final validation passed.")
print("Final train matrix:", X_train_ready.shape)
print("Final test matrix:", X_test_ready.shape)

## 18. Export the ML-ready data (optional)

The transformed matrices can be saved for downstream experiments. Save the fitted pipeline as well so that future raw records receive exactly the same transformations. In production, prefer persisting the pipeline and applying it to incoming raw data rather than independently preprocessing each batch.

In [ ]:
import joblib
# Uncomment to save files in the notebook's working directory:
# X_train_ready.assign(churn=y_train).to_csv("train_ml_ready.csv", index=False)
# X_test_ready.assign(churn=y_test).to_csv("test_ml_ready.csv", index=False)
# joblib.dump(complete_pipeline, "complete_preprocessing_model.joblib")